In [57]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.utils import resample
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, TimeDistributed
from tensorflow.keras.utils import to_categorical

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_PATH = '.../Dataset'
CSV_PATH  = os.path.join(BASE_PATH, 'wustl-scada-2018.csv')
OUT_PATH  = BASE_PATH

print(f'CSV path : {CSV_PATH}')
print(f'Exists   : {os.path.exists(CSV_PATH)}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CSV path : /content/drive/MyDrive/Dataset/wustl-scada-2018.csv
Exists   : True


In [59]:
df = pd.read_csv(CSV_PATH, low_memory=False)
for col in df.select_dtypes([object]):
    df[col] = df[col].str.decode('utf-8')

print(f'Shape : {df.shape}')
print(f'Last column: {df.columns[-1]}')
print(df[df.columns[-1]].value_counts())

Shape : (7037983, 7)
Last column: Target
Target
0    6634581
1     403402
Name: count, dtype: int64


In [60]:
selected_features = [ 'TotPkts', 'TotBytes','DstPkts']

df = df[selected_features + [df.columns[-1]]].dropna()
df.reset_index(drop=True, inplace=True)

print(f'Shape after dropna : {df.shape}')

Shape after dropna : (7037983, 4)


In [61]:
X_raw = df[selected_features].values
y_raw = df[df.columns[-1]].values

scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

In [62]:
def ensure_3d_X(X):
    X = np.asarray(X)
    if X.ndim == 2:
        X = X.astype(np.float32)[..., np.newaxis]
    elif X.ndim == 3:
        X = X.astype(np.float32)
    else:
        raise ValueError(f'X must be 2D/3D. Got: {X.shape}')
    return X

def ensure_y_shape(y):
    y = np.asarray(y)
    if y.dtype == object:
        le = LabelEncoder()
        y = le.fit_transform(y)
    if not np.issubdtype(y.dtype, np.integer):
        y = y.astype(int)
    classes = np.unique(y)
    if len(classes) == 2 and set(classes) <= {0, 1}:
        y = y.reshape(-1, 1).astype(np.float32)
    else:
        y = to_categorical(y.astype(int)).astype(np.float32)
    return y

X = ensure_3d_X(X_scaled)
y = ensure_y_shape(y_raw)

print('After fix:', X.shape, X.dtype, '|', y.shape, y.dtype)
print('Unique y classes:', int(y.shape[1]) if y.ndim == 2 else 1, 'class(es)')

After fix: (7037983, 3, 1) float32 | (7037983, 1) float32
Unique y classes: 1 class(es)


In [63]:
print(f"df shape       : {df.shape}")
print(f"df duplicated  : {df.duplicated().sum()}")
print(f"X_scaled shape : {X_scaled.shape}")
print(f"label shape    : {df['Target'].values.shape}")

df shape       : (7037983, 4)
df duplicated  : 7037023
X_scaled shape : (7037983, 3)
label shape    : (7037983,)


In [64]:
class JaspenCorrelation(tf.keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def call(self, t, r):
        t_exp  = tf.expand_dims(t, 1)
        dot    = tf.reduce_sum(t_exp * r, axis=-1)
        norm_t = tf.norm(t_exp, axis=-1)
        norm_r = tf.norm(r,     axis=-1)
        corr   = dot / (norm_t * norm_r + 1e-8)
        corr   = tf.maximum(corr, 0.0)
        return tf.reduce_mean(corr, axis=1, keepdims=True)

    def get_config(self):
        return super().get_config()


def create_model(ref_count, seq_len, channels, num_classes, is_multiclass):
    test_input      = Input(shape=(seq_len, channels), name='test_input')
    reference_input = Input(shape=(ref_count, seq_len, channels), name='reference_input')

    # seq_len=6: padding='same' dan 1 MaxPooling saja
    shared_cnn = tf.keras.Sequential([
        Conv1D(64, 3, activation='relu', padding='same'),
        MaxPooling1D(2),
        Conv1D(128, 3, activation='relu', padding='same'),
        Flatten()
    ], name='shared_cnn')

    test_encoded      = shared_cnn(test_input)
    reference_encoded = TimeDistributed(shared_cnn)(reference_input)

    agg = JaspenCorrelation()(test_encoded, reference_encoded)

    x = Dense(128, activation='relu')(agg)

    if is_multiclass:
        output = Dense(num_classes, activation='softmax')(x)
        loss   = 'categorical_crossentropy'
    else:
        output = Dense(num_classes, activation='sigmoid')(x)
        loss   = 'binary_crossentropy'

    model = Model(inputs=[test_input, reference_input], outputs=output)
    return model, loss

In [65]:
def calculate_all_metrics(y_true, y_pred, is_multiclass, threshold=0.5):
    if is_multiclass:
        y_true_cls = np.argmax(y_true, axis=1)
        y_pred_cls = np.argmax(y_pred, axis=1)
        acc  = accuracy_score(y_true_cls, y_pred_cls)
        prec = precision_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
        rec  = recall_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
        f1   = f1_score(y_true_cls, y_pred_cls, average='macro', zero_division=0)
    else:
        y_pred_bin = (y_pred >= threshold).astype(int)
        average    = 'binary' if y_true.shape[1] == 1 else 'macro'
        acc  = accuracy_score(y_true, y_pred_bin)
        prec = precision_score(y_true, y_pred_bin, average=average, zero_division=0)
        rec  = recall_score(y_true, y_pred_bin, average=average, zero_division=0)
        f1   = f1_score(y_true, y_pred_bin, average=average, zero_division=0)
    return {
        'Accuracy' : acc  * 100.0,
        'Precision': prec * 100.0,
        'Recall'   : rec  * 100.0,
        'F1_Score' : f1   * 100.0
    }

def calculate_space_per_instance_broadcast(X_sample, y_sample):
    return (X_sample.nbytes + y_sample.nbytes) / (1024 * 1024)

def reference_set_size_mb(reference_set):
    return reference_set.nbytes / (1024 * 1024)

def measure_single_instance_time(model, X_sample, reference_set, num_trials=5):
    ref_input = np.broadcast_to(reference_set[np.newaxis, ...], (1,) + reference_set.shape)
    _ = model.predict([X_sample, ref_input], verbose=0)
    times = []
    for _ in range(num_trials):
        t0 = time.time()
        _ = model.predict([X_sample, ref_input], verbose=0)
        times.append((time.time() - t0) * 1000.0)
    return float(np.mean(times))

In [66]:
def undersample(X, y, random_state=42):
    if y.ndim == 2 and y.shape[1] == 1:
        y_1d = y.ravel().astype(int)
    else:
        y_1d = np.argmax(y, axis=1)

    classes, counts = np.unique(y_1d, return_counts=True)
    min_count = counts.min()

    X_parts, y_parts = [], []
    for cls in classes:
        idx = np.where(y_1d == cls)[0]
        idx_sampled = resample(idx, n_samples=min_count,
                               replace=False, random_state=random_state)
        X_parts.append(X[idx_sampled])
        y_parts.append(y[idx_sampled])

    X_bal = np.concatenate(X_parts, axis=0)
    y_bal = np.concatenate(y_parts, axis=0)

    perm  = np.random.RandomState(random_state).permutation(len(X_bal))
    return X_bal[perm], y_bal[perm]

In [67]:
def run_full_data(X, y, ref_count_fixed=16, epochs=10, batch_size=16,
                  test_size=0.2, random_state=42):
    assert X.shape[0] == y.shape[0]
    N, SEQ_LEN, CHANNELS = X.shape[0], X.shape[1], X.shape[2]
    NUM_CLASSES   = y.shape[1]
    is_multiclass = (
        (NUM_CLASSES > 1) and
        np.all((y == 0) | (y == 1)) and
        np.all(y.sum(axis=1) == 1)
    )

    X = X.astype(np.float32)
    y = y.astype(np.float32)

    print('=' * 70)
    print(f'Running on full data: N = {N:,}')
    print('=' * 70)

    if is_multiclass:
        y_strat = np.argmax(y, axis=1)
    else:
        y_strat = y.squeeze()

    X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
        X, y, test_size=test_size, stratify=y_strat, random_state=random_state
    )

    # Undersample train set
    X_train_s, y_train_s = undersample(X_train_s, y_train_s, random_state=random_state)
    print(f'After undersampling: X_train={X_train_s.shape}, y_train={y_train_s.shape}')
    print(f'Test set           : X_test={X_test_s.shape}, y_test={y_test_s.shape}')

    ref_count     = min(ref_count_fixed, len(X_train_s))
    reference_set = X_train_s[:ref_count]

    ref_train = np.broadcast_to(
        reference_set[np.newaxis, ...], (X_train_s.shape[0],) + reference_set.shape
    )
    ref_test = np.broadcast_to(
        reference_set[np.newaxis, ...], (X_test_s.shape[0],) + reference_set.shape
    )

    model, loss = create_model(ref_count, SEQ_LEN, CHANNELS, NUM_CLASSES, is_multiclass)
    model.compile(optimizer='adam', loss=loss, metrics=['accuracy'])

    X_sample = X_train_s[0:1]
    y_sample = y_train_s[0:1]
    space_per_instance = calculate_space_per_instance_broadcast(X_sample, y_sample)
    S_com = N * space_per_instance + reference_set_size_mb(reference_set)

    t0 = time.time()
    history=model.fit(
        [X_train_s, ref_train], y_train_s,
        validation_data=([X_test_s, ref_test], y_test_s),
        epochs=epochs, batch_size=batch_size, verbose=1
    )
    train_time = time.time() - t0

    pred_start      = time.time()
    y_pred          = model.predict([X_test_s, ref_test], verbose=0)
    metrics         = calculate_all_metrics(y_test_s, y_pred, is_multiclass)
    prediction_time = time.time() - pred_start

    tpi   = measure_single_instance_time(model, X_sample, reference_set, num_trials=5)
    T_com = N * tpi

    print(f'\nREF_COUNT used           : {ref_count}')
    print(f'Train size               : {X_train_s.shape[0]:,}')
    print(f'Test size                : {X_test_s.shape[0]:,}')
    print(f'Train time (s)           : {train_time:.2f}')
    print(f'Acc / Prec / Rec / F1 (%): '
          f'{metrics["Accuracy"]:.2f} / {metrics["Precision"]:.2f} / '
          f'{metrics["Recall"]:.2f} / {metrics["F1_Score"]:.2f}')
    print(f'T_com (ms)               : {T_com:.2f}')
    print(f'S_com (MB)               : {S_com:.2f}')
    print(f'Prediction time (s)      : {prediction_time:.2f}')
    print(f'history                   : {history:.2f}')

    return {
        'N'                  : N,
        'ref_count'          : ref_count,
        'accuracy'           : metrics['Accuracy'],
        'precision'          : metrics['Precision'],
        'recall'             : metrics['Recall'],
        'f1_score'           : metrics['F1_Score'],
        'training_time'      : train_time,
        'time_complexity_ms' : T_com,
        'space_complexity_mb': S_com,
        'prediction_time'    : prediction_time,
        'model'              : model,
        'history'            : history
    }

In [68]:
print(X.shape, y.shape, X.dtype, y.dtype)

results = run_full_data(
    X, y,
    ref_count_fixed=20,
    epochs=10,
    batch_size=16
)

(7037983, 3, 1) (7037983, 1) float32 float32
Running on full data: N = 7,037,983
After undersampling: X_train=(645444, 3, 1), y_train=(645444, 1)
Test set           : X_test=(1407597, 3, 1), y_test=(1407597, 1)
Epoch 1/10
40341/40341 ━━━━━━━━━━━━━━━━━━━━ 368s 9ms/step - accuracy: 0.9846 - loss: 0.0579 - val_accuracy: 0.9925 - val_loss: 0.0486
Epoch 2/10
40341/40341 ━━━━━━━━━━━━━━━━━━━━ 353s 9ms/step - accuracy: 0.9936 - loss: 0.0390 - val_accuracy: 0.9925 - val_loss: 0.0375
Epoch 3/10
40341/40341 ━━━━━━━━━━━━━━━━━━━━ 353s 9ms/step - accuracy: 0.9936 - loss: 0.0391 - val_accuracy: 0.9925 - val_loss: 0.0395
Epoch 4/10
40341/40341 ━━━━━━━━━━━━━━━━━━━━ 352s 9ms/step - accuracy: 0.9936 - loss: 0.0388 - val_accuracy: 0.9925 - val_loss: 0.0381
Epoch 5/10
40341/40341 ━━━━━━━━━━━━━━━━━━━━ 351s 9ms/step - accuracy: 0.9936 - loss: 0.0388 - val_accuracy: 0.9925 - val_loss: 0.0422
Epoch 6/10
40341/40341 ━━━━━━━━━━━━━━━━━━━━ 349s 9ms/step - accuracy: 0.9936 - loss: 0.0388 - val_accuracy: 0.9925 - va

TypeError: unsupported format string passed to History.__format__

In [ ]:
print('\n' + '='*80)
print('Final Results')
print('='*80)
print(f"Total Data (N)      : {results['N']:,}")
print(f"Ref Count           : {results['ref_count']}")
print(f"Accuracy (%)        : {results['accuracy']:.2f}")
print(f"Precision (%)       : {results['precision']:.2f}")
print(f"Recall (%)          : {results['recall']:.2f}")
print(f"F1 Score (%)        : {results['f1_score']:.2f}")
print(f"Training Time (s)   : {results['training_time']:.2f}")
print(f"T_com (ms)          : {results['time_complexity_ms']:.2f}")
print(f"S_com (MB)          : {results['space_complexity_mb']:.2f}")
print(f"Prediction Time (s) : {results['prediction_time']:.2f}")